# IEDA 3130 · Pose Estimation Demo (Colab version)
HKUST IEDA 3130 Ergonomics and Safety Management · TA: Junjie HOU

Run the cells top-to-bottom (Shift+Enter). No local installation needed. If a cell fails, **Runtime → Restart session** and run again.

In [ ]:
!pip -q install mediapipe opencv-python-headless
!git clone -q https://github.com/Junjie31/ieda3130-pose-demo.git
%cd ieda3130-pose-demo
import mediapipe, cv2; print('mediapipe', mediapipe.__version__, '| opencv', cv2.__version__)

## 1 · Upload a photo
Side view, whole body, plain background (see README §3). You may also use the sample fitness image below if you have no photo.

In [ ]:
from google.colab import files
up = files.upload()
IMG = next(iter(up)) if up else None
if IMG is None:
    !wget -q -O sample.jpg https://cdn.pixabay.com/photo/2019/03/12/20/39/girl-4051811_960_720.jpg
    IMG = 'sample.jpg'
print('using', IMG)

## 2 · Run the demo
Add `--known shoulder_elbow=33.5` (your tape measurement in cm) to get the `calibrated_cm` column.

In [ ]:
!python pose_demo.py --image "$IMG" --no-show --model full

In [ ]:
import os, pandas as pd
from IPython.display import Image, display
stem = os.path.splitext(os.path.basename(IMG))[0]
display(Image(f'outputs/{stem}_pose.jpg', width=480))
pd.read_csv(f'outputs/{stem}_measures.csv').fillna('')

## 3 · Webcam snapshot (optional)
Colab cannot stream video, but it can take one photo from your browser camera.

In [ ]:
from IPython.display import Javascript
from google.colab.output import eval_js
from base64 import b64decode
js = Javascript('''
async function takePhoto() {
  const div = document.createElement('div');
  const video = document.createElement('video'); video.style.display='block';
  const stream = await navigator.mediaDevices.getUserMedia({video: true});
  document.body.appendChild(div); div.appendChild(video); video.srcObject = stream; await video.play();
  const btn = document.createElement('button'); btn.textContent = 'Capture'; div.appendChild(btn);
  await new Promise(r => btn.onclick = r);
  const c = document.createElement('canvas'); c.width = video.videoWidth; c.height = video.videoHeight;
  c.getContext('2d').drawImage(video, 0, 0); stream.getVideoTracks()[0].stop(); div.remove();
  return c.toDataURL('image/jpeg', 0.9);
}''')
display(js)
data = eval_js('takePhoto()')
open('webcam.jpg','wb').write(b64decode(data.split(',')[1]))
!python pose_demo.py --image webcam.jpg --no-show
display(Image('outputs/webcam_pose.jpg', width=480))

## 4 · Download your results
Fill in `WORKSHEET.md` with the numbers above and compare with your Lab 2 Excel sheet.

In [ ]:
!zip -q -r outputs.zip outputs
files.download('outputs.zip')